# PTM Binder Design with Artisan

This notebook mirrors the phosphopeptide binder-design tutorial style, but it keeps the actual model work inside `ptm_foundry` and uses `artisan` only for pipeline orchestration.

We will run the same workshop target used in the Colab notebook:

- target peptide: `PVPNPD(PTR)EPIRKGQ`
- target chain: `B`
- binder chain: `A`
- stages: spoof target -> RFD3 -> LigandMPNN -> RF3 -> tutorial-style metrics -> filter
        


## 0. Setup

This notebook works in both Google Colab and a local PTM Workshop / Pixi setup.

- In Colab: switch to a GPU runtime and run the setup cells from the top.
- Locally: launch Jupyter from the `ptm_foundry` repo in the `PTM Workshop` kernel or any kernel backed by the Pixi `dev` environment.
- If `artisan` is not already checked out next to `ptm_foundry`, the setup cell below can clone it automatically.

Recommended local setup from the repo root:

```bash
pixi install -e dev
pixi run -e dev install-workshop-kernel
pixi run -e dev workshop-notebook
```

The setup cells below will:

- detect or clone `ptm_foundry` and `artisan`
- install missing Python dependencies for the current kernel
- download the default RFD3 / LigandMPNN / RF3 checkpoints if they are missing
- start a local Prefect server automatically for local sessions when one is available
        


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore", module="atomworks")

IN_COLAB = "google.colab" in sys.modules
PTM_FOUNDRY_GIT_URL = os.environ.get(
    "PTM_FOUNDRY_GIT_URL",
    "https://github.com/magnusbauer/ptm_foundry.git",
)
PTM_FOUNDRY_GIT_REF = os.environ.get("PTM_FOUNDRY_GIT_REF", "production")
ARTISAN_GIT_URL = os.environ.get(
    "ARTISAN_GIT_URL",
    "https://github.com/dexterity-systems/artisan.git",
)
ARTISAN_GIT_REF = os.environ.get("ARTISAN_GIT_REF", "release/v0.1.2a2")


def clone_repo(url: str, ref: str, dest: Path) -> None:
    if dest.exists():
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "clone", "--branch", ref, url, str(dest)])


if IN_COLAB:
    REPO_DIR = Path("/content/foundry")
    clone_repo(PTM_FOUNDRY_GIT_URL, PTM_FOUNDRY_GIT_REF, REPO_DIR)
else:
    cwd = Path.cwd().resolve()
    REPO_DIR = next(
        (
            candidate
            for candidate in [cwd, *cwd.parents]
            if (candidate / "examples").exists() and (candidate / "models").exists()
        ),
        cwd,
    )

ARTISAN_REPO = Path(
    os.environ.get("ARTISAN_REPO", str(REPO_DIR.parent / "artisan"))
).expanduser().resolve()
AUTO_CLONE_ARTISAN = os.environ.get("ARTISAN_AUTO_CLONE", "1") == "1"
if not ARTISAN_REPO.exists():
    if not AUTO_CLONE_ARTISAN:
        raise FileNotFoundError(
            f"Artisan repo not found at {ARTISAN_REPO}. "
            "Set ARTISAN_REPO or ARTISAN_AUTO_CLONE=1."
        )
    clone_repo(ARTISAN_GIT_URL, ARTISAN_GIT_REF, ARTISAN_REPO)

EXAMPLES_DIR = REPO_DIR / "examples"

SRC_PATHS = [
    REPO_DIR / "src",
    REPO_DIR / "models" / "rfd3" / "src",
    REPO_DIR / "models" / "mpnn" / "src",
    REPO_DIR / "models" / "rf3" / "src",
    EXAMPLES_DIR,
    ARTISAN_REPO / "src",
]
for path in SRC_PATHS:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f"Running in Colab: {IN_COLAB}")
print(f"Repository root: {REPO_DIR}")
print(f"Artisan repo:    {ARTISAN_REPO}")
if IN_COLAB:
    print(f"PTM Foundry ref: {PTM_FOUNDRY_GIT_URL} @ {PTM_FOUNDRY_GIT_REF}")
print(f"Artisan ref:     {ARTISAN_GIT_URL} @ {ARTISAN_GIT_REF}")
        


Running in Colab: False
Repository root: /net/scratch/magnusb/43_workshop/ptm_foundry
Artisan repo:    /net/scratch/magnusb/43_workshop/artisan
Artisan ref:     https://github.com/dexterity-systems/artisan.git @ release/v0.1.2a2


In [2]:
%%time

import importlib.util
import sysconfig
import time
import subprocess
from urllib.request import urlretrieve

os.environ["CCD_MIRROR_PATH"] = ""
os.environ["PDB_MIRROR_PATH"] = ""

for env_name, default_value in {
    "DEBUG": "0",
    "TYPE_CHECK": "0",
    "NAN_CHECK": "1",
    "DISABLE_CUEQUIVARIANCE": "0",
}.items():
    if not os.environ.get(env_name, "").strip():
        os.environ[env_name] = default_value

CKPT_DIR = Path.home() / ".foundry" / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["FOUNDRY_CHECKPOINTS_DIR"] = str(CKPT_DIR)

PREFECT_NOTEBOOK_HOME = (REPO_DIR / ".prefect-notebook").resolve()
PREFECT_NOTEBOOK_HOME.mkdir(parents=True, exist_ok=True)
PREFECT_NOTEBOOK_PROFILES_PATH = PREFECT_NOTEBOOK_HOME / "profiles.toml"
PREFECT_HOST = os.environ.get("PTM_ARTISAN_PREFECT_HOST", "127.0.0.1")
PREFECT_PORT = int(os.environ.get("PTM_ARTISAN_PREFECT_PORT", "4311"))


def ensure_pip() -> None:
    if importlib.util.find_spec("pip") is None:
        subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])


def pip_install(packages: list[str]) -> None:
    if not packages:
        return
    ensure_pip()
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


def download_file(url: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp_dest = dest.with_suffix(dest.suffix + ".part")
    urlretrieve(url, tmp_dest)
    tmp_dest.replace(dest)


def normalize_api_url(url: str) -> str:
    url = url.rstrip("/")
    if url.endswith("/api"):
        return url
    return f"{url}/api"


PREFECT_DEFAULT_API_URL = normalize_api_url(
    os.environ.get("PTM_ARTISAN_PREFECT_API_URL", f"http://{PREFECT_HOST}:{PREFECT_PORT}")
)


def find_script(name: str) -> str | None:
    candidates = [
        shutil.which(name),
        str(Path(sysconfig.get_path("scripts")) / name),
        str(Path(sys.executable).resolve().parent / name),
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return candidate
    return None


def clear_prefect_env() -> None:
    os.environ.pop("PREFECT_API_URL", None)
    os.environ.pop("PREFECT_SUBMITIT_SERVER", None)


def configure_prefect_environment() -> None:
    PREFECT_NOTEBOOK_PROFILES_PATH.write_text(
        'active = "ptm-notebook"\n\n'
        '[profiles.ptm-notebook]\n'
        'PREFECT_SERVER_ALLOW_EPHEMERAL_MODE = "true"\n'
    )
    os.environ["PREFECT_PROFILES_PATH"] = str(PREFECT_NOTEBOOK_PROFILES_PATH)
    os.environ["PREFECT_PROFILE"] = "ptm-notebook"
    os.environ["PREFECT_SERVER_ALLOW_EPHEMERAL_MODE"] = "true"
    clear_prefect_env()


def ensure_prefect_server(*, auto_start: bool) -> str | None:
    from prefect_submitit.server.discovery import health_check

    configure_prefect_environment()
    if health_check(PREFECT_DEFAULT_API_URL):
        os.environ["PREFECT_API_URL"] = PREFECT_DEFAULT_API_URL
        return PREFECT_DEFAULT_API_URL

    if not auto_start:
        return None

    prefect_cli = find_script("prefect")
    if prefect_cli is None:
        print("prefect CLI not found; continuing without a Prefect API.")
        return None

    print(f"Starting local Prefect server at {PREFECT_DEFAULT_API_URL}...")
    try:
        subprocess.check_call([
            prefect_cli,
            "server",
            "start",
            "--host",
            PREFECT_HOST,
            "--port",
            str(PREFECT_PORT),
            "--background",
            "--no-ui",
        ])
    except subprocess.CalledProcessError as exc:
        print(f"Prefect auto-start failed ({exc}); continuing without a Prefect API.")
        clear_prefect_env()
        return None
    for _ in range(60):
        time.sleep(1)
        if health_check(PREFECT_DEFAULT_API_URL):
            os.environ["PREFECT_API_URL"] = PREFECT_DEFAULT_API_URL
            return PREFECT_DEFAULT_API_URL

    print(
        f"Prefect server at {PREFECT_DEFAULT_API_URL} did not become reachable within 60 seconds."
    )
    clear_prefect_env()
    return None


if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"],
        check=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

REQUIRED_PACKAGES = {
    "matplotlib": "matplotlib",
    "networkx": "networkx",
    "pandas": "pandas",
    "hydride": "hydride",
    "biotite": "biotite",
    "atomworks": "atomworks[ml]>=2.1.1",
    "lightning": "lightning>=2.5.0",
    "rootutils": "rootutils>=1.0.7,<1.1",
    "hydra": "hydra-core>=1.3.0,<1.4",
    "environs": "environs>=11.0.0,<12",
    "rich": "rich>=13.9.4",
    "jaxtyping": "jaxtyping>=0.2.17,<1",
    "beartype": "beartype>=0.18.0,<1",
    "loralib": "loralib>=0.1.1",
    "einops": "einops>=0.8.0,<1",
    "einx": "einx>=0.1.0,<1",
    "opt_einsum": "opt_einsum>=3.4.0,<4",
    "tree": "dm-tree>=0.1.6,<1",
    "zstandard": "zstandard",
    "toolz": "toolz",
    "pydantic": "pydantic>=2.8",
    "prefect": "prefect[dask]>=3.6,<4.0",
    "prefect_submitit": "prefect-submitit>=0.1.4",
    "submitit": "submitit",
    "asyncpg": "asyncpg",
    "deltalake": "deltalake>=0.14.0",
    "polars": "polars>=0.20.0",
    "xxhash": "xxhash",
    "ipywidgets": "ipywidgets",
}
missing_packages = [
    package
    for module_name, package in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    print("Installing missing packages:")
    for package in missing_packages:
        print(" -", package)
    pip_install(missing_packages)
else:
    print("Python dependencies already satisfied.")

CHECKPOINTS = {
    "rfd3": {
        "url": "https://files.ipd.uw.edu/pub/rfd3/rfd3_foundry_2025_12_01_remapped.ckpt",
        "filename": "rfd3_latest.ckpt",
    },
    "ligandmpnn": {
        "url": "https://files.ipd.uw.edu/pub/ligandmpnn/ligandmpnn_v_32_010_25.pt",
        "filename": "ligandmpnn_v_32_010_25.pt",
    },
    "rf3": {
        "url": "https://files.ipd.uw.edu/pub/rf3/rf3_foundry_01_24_latest_remapped.ckpt",
        "filename": "rf3_foundry_01_24_latest_remapped.ckpt",
    },
}

for name, info in CHECKPOINTS.items():
    dest = CKPT_DIR / info["filename"]
    if dest.exists():
        print(f"{name}: already present at {dest}")
        continue
    print(f"Downloading {name} -> {dest}")
    download_file(info["url"], dest)

AUTO_START_PREFECT = os.environ.get("PTM_ARTISAN_AUTO_START_PREFECT", "1") == "1"
prefect_api_url = ensure_prefect_server(auto_start=AUTO_START_PREFECT)
if prefect_api_url is None:
    raise RuntimeError(
        "Could not reach a Prefect API for PipelineManager local execution. "
        "Set PTM_ARTISAN_AUTO_START_PREFECT=1 or start one manually with "
        f"`prefect server start --host {PREFECT_HOST} --port {PREFECT_PORT} --background --no-ui`."
    )

print(f"prefect-profiles:{PREFECT_NOTEBOOK_PROFILES_PATH}")
print(f"prefect-profile: {os.environ.get('PREFECT_PROFILE', '<not set>')}")
print(f"prefect:         {find_script('prefect')}")
print(f"prefect-server:  {find_script('prefect-server')}")
print(f"PREFECT_API_URL: {prefect_api_url}")
print("\nCheckpoint directory contents:")
for checkpoint_path in sorted(CKPT_DIR.iterdir()):
    print(" -", checkpoint_path.name)
        


Python dependencies already satisfied.
rfd3: already present at /home/magnusb/.foundry/checkpoints/rfd3_latest.ckpt
ligandmpnn: already present at /home/magnusb/.foundry/checkpoints/ligandmpnn_v_32_010_25.pt
rf3: already present at /home/magnusb/.foundry/checkpoints/rf3_foundry_01_24_latest_remapped.ckpt
Ignoring unreachable Prefect API URL: http://ridgefield.dhcp.ipd:4309/api
pg_ctl not found; skipping Prefect auto-start and continuing without a Prefect API.
prefect:         /net/scratch/magnusb/43_workshop/ptm_foundry/.pixi/envs/dev/bin/prefect
prefect-server:  /net/scratch/magnusb/43_workshop/ptm_foundry/.pixi/envs/dev/bin/prefect-server
PREFECT_API_URL: <not set>
Local-only execution will proceed without a Prefect API.

Checkpoint directory contents:
 - ligandmpnn_v_32_010_25.pt
 - proteinmpnn_v_48_020.pt
 - rf3_foundry_01_24_latest_remapped.ckpt
 - rfd3_latest.ckpt
CPU times: user 2.96 s, sys: 196 ms, total: 3.16 s
Wall time: 2.63 s


In [3]:
import json

import numpy as np
import pandas as pd
from IPython.display import display
from lightning.fabric import seed_everything

from artisan.operations.curator import Filter
from artisan.orchestration import Backend, PipelineManager
from artisan.schemas.artifact.registry import ArtifactTypeDef
from artisan.visualization.inspect import inspect_metrics, inspect_pipeline
from atomworks.io.utils.visualize import view
from foundry.inference_engines.checkpoint_registry import REGISTERED_CHECKPOINTS
from spoof_cif import (
    bond_table_for_residue,
    chain_summary,
    plot_local_atom_bond_graph,
    plot_residue_bond_graph,
)
from ptm_artisan_ops import (
    BinderAlignedRMSD,
    PhosphositeHBondMetrics,
    RunLigandMPNNDesign,
    RunRFD3Design,
    RunRF3Refold,
    SelectionSASAMetrics,
    SpoofPTMTarget,
    load_atom_array,
    resolve_file_ref_paths,
)

for key in ["rfd3", "ligandmpnn", "rf3"]:
    ckpt_path = REGISTERED_CHECKPOINTS[key].get_default_path()
    print(f"{key:12s} -> {ckpt_path} (exists={Path(ckpt_path).exists()})")
        


03:03:20.663 | WARNING | foundry - DEBUG='release' is not a valid boolean; using default False

03:03:20.931 | DEBUG   | transforms - Debug mode is on

rfd3         -> /home/magnusb/.foundry/checkpoints/rfd3_latest.ckpt (exists=True)
ligandmpnn   -> /home/magnusb/.foundry/checkpoints/ligandmpnn_v_32_010_25.pt (exists=True)
rf3          -> /home/magnusb/.foundry/checkpoints/rf3_foundry_01_24_latest_remapped.ckpt (exists=True)


## 1. Configure the Workshop Run

These parameters mirror the Colab workshop notebook, but we now materialize all pipeline outputs under `examples/runs/ptm_artisan_pipeline/` so the file-reference artifacts remain usable after each step finishes.
        


In [4]:
seed_everything(7)

TARGET_SEQUENCE = "PVPNPD(PTR)EPIRKGQ"
TARGET_CHAIN_ID = "B"
BINDER_CHAIN_ID = "A"
BINDER_LENGTH = 100
EXAMPLE_NAME = "pvpnpd_ptr_workshop"
WORKSHOP_OUTPUT_DIR = REPO_DIR / "examples" / "workshop_outputs"
REUSE_WORKSHOP_EXAMPLE_NAME = "ptr_workshop"
REUSE_WORKSHOP_SETUP = all(
    (WORKSHOP_OUTPUT_DIR / f"{REUSE_WORKSHOP_EXAMPLE_NAME}{suffix}").exists()
    for suffix in (".cif", ".json")
)
PIPELINE_NAME = "ptm_artisan_pipeline"

RUNS_DIR = REPO_DIR / "examples" / "runs" / PIPELINE_NAME
DELTA_ROOT = RUNS_DIR / "delta"
STAGING_ROOT = RUNS_DIR / "staging"
WORKING_ROOT = RUNS_DIR / "working"
MATERIALIZED_ROOT = RUNS_DIR / "materialized"

SPOOF_OUTPUT_DIR = MATERIALIZED_ROOT / "spoof_target"
RFD3_OUTPUT_DIR = MATERIALIZED_ROOT / "rfd3"
MPNN_OUTPUT_DIR = MATERIALIZED_ROOT / "mpnn"
RF3_OUTPUT_DIR = MATERIALIZED_ROOT / "rf3"

for path in [
    DELTA_ROOT,
    STAGING_ROOT,
    WORKING_ROOT,
    SPOOF_OUTPUT_DIR,
    RFD3_OUTPUT_DIR,
    MPNN_OUTPUT_DIR,
    RF3_OUTPUT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

RFD3_PARAMS = {
    "ckpt_path": "rfd3",
    "diffusion_batch_size": 1,
    "n_batches": 1,
    "output_dir": str(RFD3_OUTPUT_DIR),
}
MPNN_PARAMS = {
    "checkpoint_path": "ligandmpnn",
    "batch_size": 4,
    "designed_chains": [BINDER_CHAIN_ID],
    "target_chain_id": TARGET_CHAIN_ID,
    "remove_waters": True,
    "is_legacy_weights": True,
    "output_dir": str(MPNN_OUTPUT_DIR),
}
RF3_PARAMS = {
    "ckpt_path": "rf3",
    "annotate_b_factor_with_plddt": True,
    "output_dir": str(RF3_OUTPUT_DIR),
}
FILTER_CRITERIA = [
    {"metric": "whole_peptide_ca_rmsd", "operator": "lt", "value": 2.0},
    {"metric": "po4.fraction_buried", "operator": "gt", "value": 0.85},
    {"metric": "phosphosite_hbonds", "operator": "ge", "value": 0.0},
]

print(f"Target sequence: {TARGET_SEQUENCE}")
print(f"Run directory:   {RUNS_DIR}")
print(f"Reuse workshop setup: {REUSE_WORKSHOP_SETUP}")
if REUSE_WORKSHOP_SETUP:
    print(
        f"Workshop source:    {WORKSHOP_OUTPUT_DIR / (REUSE_WORKSHOP_EXAMPLE_NAME + '.cif')}"
    )
print(json.dumps({
    "rfd3": RFD3_PARAMS,
    "mpnn": MPNN_PARAMS,
    "rf3": RF3_PARAMS,
    "filter": FILTER_CRITERIA,
}, indent=2))
        


Seed set to 7


Target sequence: PVPNPD(PTR)EPIRKGQ
Run directory:   /net/scratch/magnusb/43_workshop/ptm_foundry/examples/runs/ptm_artisan_pipeline
Reuse workshop setup: True
Workshop source:    /net/scratch/magnusb/43_workshop/ptm_foundry/examples/workshop_outputs/ptr_workshop.cif
{
  "rfd3": {
    "ckpt_path": "rfd3",
    "diffusion_batch_size": 1,
    "n_batches": 1,
    "output_dir": "/net/scratch/magnusb/43_workshop/ptm_foundry/examples/runs/ptm_artisan_pipeline/materialized/rfd3"
  },
  "mpnn": {
    "checkpoint_path": "ligandmpnn",
    "batch_size": 4,
    "designed_chains": [
      "A"
    ],
    "target_chain_id": "B",
    "remove_waters": true,
    "is_legacy_weights": true,
    "output_dir": "/net/scratch/magnusb/43_workshop/ptm_foundry/examples/runs/ptm_artisan_pipeline/materialized/mpnn"
  },
  "rf3": {
    "ckpt_path": "rf3",
    "annotate_b_factor_with_plddt": true,
    "output_dir": "/net/scratch/magnusb/43_workshop/ptm_foundry/examples/runs/ptm_artisan_pipeline/materialized/rf3"
  },

## 2. Create the Pipeline

This is the same top-level orchestration pattern used in the Artisan docs: create a `PipelineManager`, capture the `output` helper, then run one step per cell.
        


In [5]:
pipeline = PipelineManager.create(
    name=PIPELINE_NAME,
    delta_root=DELTA_ROOT,
    staging_root=STAGING_ROOT,
    working_root=WORKING_ROOT,
    preserve_working=False,
    backend=Backend.LOCAL,
    prefect_server=prefect_api_url,
)
output = pipeline.output
pipeline
        


03:04:42.248 | INFO    | artisan.orchestration.pipeline_manager - Pipeline 'ptm_artisan_pipeline' initialized (run_id=ptm_artisan_pipeline_20260416_100442_8e400cf9)

03:04:42.249 | INFO    | artisan.orchestration.pipeline_manager -   delta_root: /net/scratch/magnusb/43_workshop/ptm_foundry/examples/runs/ptm_artisan_pipeline/delta

03:04:42.249 | INFO    | artisan.orchestration.pipeline_manager -   staging_root: /net/scratch/magnusb/43_workshop/ptm_foundry/examples/runs/ptm_artisan_pipeline/staging

PipelineManager(name='ptm_artisan_pipeline', steps=0, delta_root=PosixPath('/net/scratch/magnusb/43_workshop/ptm_foundry/examples/runs/ptm_artisan_pipeline/delta'))

## 3. Spoof the PTM Target

This step uses the same `build_ptm_binder_workshop_inputs()` logic from `ptm_foundry/examples/spoof_cif.py`. It creates:

- a spoofed CIF for `PVPNPD(PTR)EPIRKGQ`
- the matching RFD3 JSON config
- compact metadata with the PTR residue index
        


In [6]:
spoof_step = pipeline.run(
    operation=SpoofPTMTarget,
    name="spoof_target",
    params={
        "sequence": TARGET_SEQUENCE,
        "binder_length": BINDER_LENGTH,
        "target_chain_id": TARGET_CHAIN_ID,
        "ptm_resname": "PTR",
        "example_name": EXAMPLE_NAME,
        "output_dir": str(SPOOF_OUTPUT_DIR),
        "reuse_existing_dir": str(WORKSHOP_OUTPUT_DIR) if REUSE_WORKSHOP_SETUP else None,
        "reuse_existing_name": REUSE_WORKSHOP_EXAMPLE_NAME if REUSE_WORKSHOP_SETUP else None,
    },
    backend=Backend.LOCAL,
)
spoof_step
        


03:04:43.891 | INFO    | artisan.orchestration.pipeline_manager - Step 0 (spoof_target) starting... [backend=local]

03:04:45.470 | ERROR   | artisan.orchestration.pipeline_manager - Step 0 (spoof_target) failed after 1.6s: RuntimeError: Failed to reach API at http://ridgefield.dhcp.ipd:4209/api/

StepResult(step_name='spoof_target', step_number=0, success=False, total_count=0, succeeded_count=0, failed_count=0, output_roles=frozenset(), output_types={}, duration_seconds=1.5602108985185623, metadata={'error': 'RuntimeError: Failed to reach API at http://ridgefield.dhcp.ipd:4209/api/'})

In [7]:
if not spoof_step.success:
    raise RuntimeError(
        "Spoof step failed before producing metadata: "
        + spoof_step.metadata.get("error", "unknown error")
    )
if "cif_path" not in spoof_step.metadata:
    raise KeyError(
        f"Expected 'cif_path' in spoof metadata, got keys: {sorted(spoof_step.metadata)}"
    )

spoofed_target_path = Path(spoof_step.metadata["cif_path"])
ptm_residue_id = int(spoof_step.metadata["ptm_residue_id"])
spoofed_target = load_atom_array(spoofed_target_path, hydrogen_policy="remove")

display(chain_summary(spoofed_target))
print(f"Spoofed target path: {spoofed_target_path}")
print(f"PTR residue id:      {ptm_residue_id}")
print(f"Reused workshop setup: {spoof_step.metadata.get('reused_existing', False)}")
if spoof_step.metadata.get("reused_existing"):
    print(
        "Workshop source:      "
        f"{spoof_step.metadata.get('reuse_source_dir')} / {spoof_step.metadata.get('reuse_source_name')}"
    )
view(spoofed_target)
        


RuntimeError: Spoof step failed before producing metadata: RuntimeError: Failed to reach API at http://ridgefield.dhcp.ipd:4209/api/

In [ ]:
bond_table = bond_table_for_residue(
    spoofed_target,
    chain_id=TARGET_CHAIN_ID,
    residue_id=ptm_residue_id,
    include_neighbors=True,
)
display(bond_table)
        


In [ ]:
plot_residue_bond_graph(
    spoofed_target,
    chain_id=TARGET_CHAIN_ID,
    residue_id=ptm_residue_id,
)
        


In [ ]:
plot_local_atom_bond_graph(
    spoofed_target,
    chain_id=TARGET_CHAIN_ID,
    residue_id=ptm_residue_id,
)
        


## 4. Generate Binder Backbones with RFD3

The `rfd3` step consumes the spoofed CIF plus the RFD3 JSON config and materializes one or more designed complexes.
        


In [ ]:
rfd3_step = pipeline.run(
    operation=RunRFD3Design,
    name="rfd3",
    inputs={
        "structures": output("spoof_target", "structures"),
        "config": output("spoof_target", "config"),
    },
    params=RFD3_PARAMS,
    backend=Backend.LOCAL,
)
rfd3_step
        


In [ ]:
rfd3_paths = [Path(path) for path in rfd3_step.metadata["structure_paths"]]
rfd3_complex = load_atom_array(rfd3_paths[0], hydrogen_policy="remove")

print("RFD3 outputs:")
for path in rfd3_paths:
    print(" -", path)

display(chain_summary(rfd3_complex))
display(inspect_metrics(DELTA_ROOT, rfd3_step.step_number))
view(rfd3_complex)
        


## 5. Design the Binder Sequence with LigandMPNN

This step keeps the peptide chain fixed and redesigns only binder chain `A`.
        


In [ ]:
mpnn_step = pipeline.run(
    operation=RunLigandMPNNDesign,
    name="mpnn",
    inputs={"structures": output("rfd3", "structures")},
    params=MPNN_PARAMS,
    backend=Backend.LOCAL,
)
mpnn_step
        


In [ ]:
mpnn_metrics = inspect_metrics(DELTA_ROOT, mpnn_step.step_number)
display(mpnn_metrics)

mpnn_paths = [Path(path) for path in mpnn_step.metadata["structure_paths"]]
selected_mpnn_path = mpnn_paths[0]
selected_complex = load_atom_array(selected_mpnn_path, hydrogen_policy="remove")

print(f"Selected MPNN design: {selected_mpnn_path}")
view(selected_complex)
        


## 6. Refold the Designs with RF3

The RF3 step refolds each LigandMPNN-designed complex and records the standard RF3 summary confidence metrics.
        


In [ ]:
rf3_step = pipeline.run(
    operation=RunRF3Refold,
    name="rf3",
    inputs={"structures": output("mpnn", "structures")},
    params=RF3_PARAMS,
    backend=Backend.LOCAL,
)
rf3_step
        


In [ ]:
rf3_metrics = inspect_metrics(DELTA_ROOT, rf3_step.step_number)
display(rf3_metrics)

rf3_paths = [Path(path) for path in rf3_step.metadata["structure_paths"]]
first_rf3_path = rf3_paths[0]
first_rf3_complex = load_atom_array(first_rf3_path, hydrogen_policy="remove")

print(f"First RF3 output: {first_rf3_path}")
view(first_rf3_complex)
        


## 7. Tutorial-Style Post-RF3 Metrics

These three steps reproduce the key tutorial metrics after RF3:

- binder-backbone-aligned peptide and phosphosite RMSDs
- phosphosite hydrogen bonds
- phosphate / PTR burial by SASA
        


In [ ]:
rmsd_step = pipeline.run(
    operation=BinderAlignedRMSD,
    name="rmsd_metrics",
    inputs={
        "reference": output("mpnn", "structures"),
        "mobile": output("rf3", "structures"),
    },
    params={
        "target_chain_id": TARGET_CHAIN_ID,
        "ptm_residue_id": ptm_residue_id,
        "ptm_resname": "PTR",
        "binder_chain_id": BINDER_CHAIN_ID,
    },
    backend=Backend.LOCAL,
)
rmsd_step
        


In [ ]:
rmsd_metrics = inspect_metrics(DELTA_ROOT, rmsd_step.step_number)
display(rmsd_metrics)
        


In [ ]:
hbond_step = pipeline.run(
    operation=PhosphositeHBondMetrics,
    name="hbond_metrics",
    inputs={"structures": output("rf3", "structures")},
    params={
        "target_chain_id": TARGET_CHAIN_ID,
        "ptm_residue_id": ptm_residue_id,
        "ptm_resname": "PTR",
    },
    backend=Backend.LOCAL,
)
hbond_step
        


In [ ]:
hbond_metrics = inspect_metrics(DELTA_ROOT, hbond_step.step_number)
display(hbond_metrics)
        


In [ ]:
sasa_step = pipeline.run(
    operation=SelectionSASAMetrics,
    name="sasa_metrics",
    inputs={"structures": output("rf3", "structures")},
    params={
        "target_chain_id": TARGET_CHAIN_ID,
        "ptm_residue_id": ptm_residue_id,
        "ptm_resname": "PTR",
    },
    backend=Backend.LOCAL,
)
sasa_step
        


In [ ]:
sasa_metrics = inspect_metrics(DELTA_ROOT, sasa_step.step_number)
display(sasa_metrics)
        


In [ ]:
final_metric_table = (
    inspect_metrics(DELTA_ROOT, rmsd_step.step_number)
    .join(
        inspect_metrics(DELTA_ROOT, hbond_step.step_number).select("name", "phosphosite_hbonds"),
        on="name",
        how="left",
    )
    .join(
        inspect_metrics(DELTA_ROOT, sasa_step.step_number).select(
            "name",
            "po4.fraction_buried",
            "ptr.fraction_buried",
        ),
        on="name",
        how="left",
    )
)
display(final_metric_table)
        


## 8. Filter the RF3 Outputs

This uses Artisan's built-in `Filter` operation to keep only RF3 structures that satisfy the workshop thresholds.
        


In [ ]:
filter_step = pipeline.run(
    operation=Filter,
    name="filter",
    inputs={"passthrough": output("rf3", "structures")},
    params={"criteria": FILTER_CRITERIA},
    backend=Backend.LOCAL,
)
filter_step
        


In [ ]:
finalize_summary = pipeline.finalize()
print(json.dumps(finalize_summary, indent=2))
        


## 9. Inspect the Pipeline and Final Structures

`inspect_pipeline()` gives the step-level overview. Then we resolve the final `Filter` output reference back to concrete file paths so you can open the surviving CIFs.
        


In [ ]:
inspect_pipeline(DELTA_ROOT)
        


In [ ]:
filtered_paths = resolve_file_ref_paths(DELTA_ROOT, filter_step.output("passthrough"))
filtered_df = pd.DataFrame({"filtered_cif": [str(path) for path in filtered_paths]})
display(filtered_df)
print("Filter diagnostics:")
print(json.dumps(filter_step.metadata, indent=2, default=float))
        


In [ ]:
if filtered_paths:
    final_path = filtered_paths[0]
    final_atom_array = load_atom_array(final_path, hydrogen_policy="remove")
    print(f"Viewing top surviving RF3 complex: {final_path}")
    view(final_atom_array)
else:
    print("No RF3 structures passed the final filter in this run.")
        
